# Тесты модели расчетов

In [1]:
from nrs import EType, NRS_Model, Element, NRS_Revision, NRS_Observer_E    # классы НРС
from nrs import NRS_Data as nd                                      # Табличные данные НРС
from nrs import q_out_simple, q_out_nozzle                          # модели расчета параметров
import matplotlib.pyplot as plt                                     # библиотеки для отрисовки получаемых данных

## Проблема отказа при больших разбегах невязки

Имеют место случаи, когда система может быть рассчитана, однако с использованием расчетной модели на итерационной релаксации она нерассчитываемеа

Модель:

In [ ]:
# Проводимости насадков
p_2 = NRS_Revision.calc_p(2.16, 40)
p_4 = NRS_Revision.calc_p(4.85, 40)
p_6 = NRS_Revision.calc_p(6, 40)
p_8 = NRS_Revision.calc_p(8.61, 40)
p_10 = NRS_Revision.calc_p(10, 40)


# Сопротивление одного рукава 51 мм, эмпирическое:
ss51 = 0.09

# Создаем модель НРС:
model = NRS_Model('Модель от одного насоса к одному стволу')

# Создаем элементы модели:
pump   = Element('Н1', EType.PUMP, H_add=40)                                       # Насос
hose   = Element('РРЛ Р1-1', EType.CONNECTOR, s = ss51, n=5)                   # рабочая рукавная линия
nozzle = Element('Ствол 1', EType.NOZZLE, p = p_10, q_out = q_out_nozzle)          # ствол №1


# Соединяем элементы модели вместе в НРС:
# Магистральная часть:
pump.append(hose).append(nozzle)

# Добавляем обозревателей:
NRS_Observer_E(pump, ['q', 'H_add'])             # Для насоса
NRS_Observer_E(hose, ['q', 'h']) 
NRS_Observer_E(nozzle, ['q', 'H_in'])            # Для ствола


# Строим модель:
model = model.build(pump, interpretate=True)

NRS_Revision.print_model_elements(model)

Модель от одного насоса к двум стволам через одно разветвление с добавленными сопротивлениями в РТ-80
all:
  Н1
  РРЛ Р1-1
  Ствол 1
in:
  Н1
out:
  Ствол 1


Тест расчета

In [21]:
# Активируем обозревателей
model = model.observersInit()
# Шаг моделирования
model.drop_q()
# Напор на насосе
pump.set_H_add(40)

model, r = model.calc(accuracy=0.05)
print('Производительность насоса', pump.q, 'л/с')
print('Потеря напора в рукавной линии', hose.h, 'м')
print('Напор перед стволом', nozzle.H_in, 'м, Производительность ствола', nozzle.q, 'л/с')

ValueError: Напор не может быть выше 120

In [22]:
hose.history()['h']

[0.0, 44.99999999999999]

Видно, что потеря напора в рукавной линии составляет 44,9 м, что выше чем напор на насосе, что в итоге приводит к тому, что до ствола доходит - 4,9 м (отрицательное значение), что делает невозможным расчет производительности ствола исходя из формулы $p*\sqrt{H}$.

При этом расчет требуемого напора на насосе по формулам тактики дает:

$$
H_н = H_{ств} + h_{р} = H_{ств} + s * n * Q^2 = 40 + 0.09 * 5 * 10^2 = 40 + 44.9 = 84.9 м
$$

In [24]:
# Активируем обозревателей
model = model.observersInit()
# Шаг моделирования
model.drop_q()
# Напор на насосе
pump.set_H_add(84.9)

model, r = model.calc(accuracy=0.05)
print('Производительность насоса', pump.q, 'л/с')
print('Потеря напора в рукавной линии', hose.h, 'м')
print('Напор перед стволом', nozzle.H_in, 'м, Производительность ствола', nozzle.q, 'л/с')

ValueError: Напор не может быть выше 120

In [25]:
hose.history()['h']

[0.0, 95.5125]

Но даже при установке значений полученных расчетом, все-равно появляется ошибка, вызванная слишком сильными колебаниями на итерации.

Таким образом предлагается решение - добавить фильтр разбега, снизить величину колебаний на итерации за счет уменьшения разброса в n раз.

### Исправление

In [41]:
# Вариант с коэффициентом 1/2
Hin = 40
s = 0.09
n = 5
q = 10

h = (s * n * q**2)
print('h чистый = ', h)

Ho = Hin - h
print('Напор на выходе чистый', Ho)

hd = Hin - Ho
print('Дифферент потери напора', hd)


h чистый =  44.99999999999999
Напор на выходе чистый -4.999999999999993
Дифферент потери напора 44.99999999999999


In [57]:
class Element(object):
    '''
    Класс элемента НРС.
    '''

    def __init__(self,
                 name:   str,
                 e_type: EType,
                 q:      float = 3.7,
                 s:      float = 0,
                 H_in:   float = 0,
                 h:      float = 0,
                 H_add:  float = 0,
                 z:      float = 0,
                 p:      float = 1,
                 n:      int   = 1,
                 l:      float = 0,
                 ri:     int   = 1,
                 ro:     int   = 1,
                 q_out:  float = q_out_simple,
                 ):
        '''
        # Аргументы
            `name`: String. Имя элемента

            `e_type`: тип элемента: 0 - подача (например, насос), 1 - связи (рукавные линии и оборудование), 2 - расход (например, стволы)

            `q`=3.7: стартовый расход через элемент, л/с

            `s`=0: гидравлическое сопротивление элемента

            `H_in`=40: Напор на входе в элемент, м

            `h`=0: стартовые потери напора на элементе, м

            `H_add`=0: дополнительный напор на элементе, м. Например, напор на насосе.

            `z`=0: перепад высот на элементе, м

            `p`=1: проводимость элемента. Для большинства элементов = 1, для расходов должен браться в соответствии с табличными значениями

            `n`=1: количество единиц элемента. Например, количество рукавов в рукавной линии

            `l`=0: длина единицы элемента. По умолчанию предполагается, что все элементы - точечные, т.е не имеют длины

            `ri`=1, `ro`=1: количество входов и выходов соответственно

            `q_out` = q_out_simple: функция расчета расхода на выходе из элемента
        '''
        self.elements_next=[]
        self.elements_previous=[]

        self.type = e_type
        self.name = name
        # print("Новый элемент НРС: " + self.name)

        self.q =q
        self.s=s
        self.H_in=H_in
        self.h=h
        self.z=z
        self.p=p
        self.n=n
        self.q_out=q_out
        self.H_add=H_add
        self.observer=None
        self.l = l
        self.ri = ri
        self.ro = ro
        # self.h=0

    def append(self, elmnt):
        '''
        Подключает элемент к выходу текущего
            Вход:
                `elmnt`:Element
                Элемент который подключается к текущему
            Выход:
                ссылка на текущий элемент
        '''
        # если имеется возможность подключения дополнительного элемента у текущего элемента
        if self.ro>len(self.elements_next):
            # Если у переданного элемента имеется возможность подключения дополнительного эл-та:
            if elmnt.ri>len(elmnt.elements_previous):
                self.elements_next.append(elmnt)
                elmnt.elements_previous.append(self)
            else:
                logger.debug(f'У элемента {elmnt.name} нет дополнительных входов для подключения {self.name}')
        else:
            logger.debug(f'У элемента {self.name} нет дополнительных выходов для подключения {elmnt.name}')
        return elmnt

    def addToModel(self, model):
        '''
        Включает элемент в модель
            Вход:
                model=Model: ссылка на модель в которую следует включить текущий элемент
            Выход:
                ссылка на текущий элемент
        '''
        model.appendElement(self)
        return self

    def fixState(self):
        '''
        Фиксирует состояние параметров элемента
            Выход:
                ссылка на текущий элемент
        '''
        if self.observer:
            self.observer.fix()
        return self

    def history(self):
        '''
        Возвращает историю изменений элемента
            Выход:
                список изменений (при наличии подключенного наблюдателя)
        '''
        if self.observer:
            return self.observer.history()
        else:
            return []

    def observerInit(self):
        '''
        Инициация наблюдателя (при наличии). Имеющаяся история изменений будет удалена.
        '''
        if self.observer:
            return self.observer.par_dict_init()
        return self

    def drop_links(self, linked_elements=False, current_element=True):
        '''
        Очистка связей элемента
        '''
        if linked_elements:
            for elmnt_next in self.elements_next:
                # eid = NRS_Revision.get_element_by_name(elmnt_next.elements_previous, self.name)
                # del elmnt_next.elements_previous[eid]
                if self in elmnt_next.elements_previous:
                    elmnt_next.elements_previous.remove(self)
            for elmnt_prev in self.elements_previous:
                # eid = NRS_Revision.get_element_by_name(elmnt_prev.elements_next, self.name)
                # del elmnt_prev.elements_next[eid]
                if self in elmnt_prev.elements_next:
                    elmnt_prev.elements_next.remove(self)

        if current_element:
            self.elements_next=[]
            self.elements_previous=[]

        return self

    def set_ri(self, new_val):
        '''
        Установка количества входных патрубков.
        Если в данный момент к элементу подключено больше элементов на вход, чем новое ri, то лишние отбрасываются.
        '''
        self.ri = new_val
        if self.ri<len(self.elements_previous):
            num_to_drop=len(self.elements_previous)-self.ri
            for _ in range(num_to_drop):
                pe = self.elements_previous.pop(0)
                pe.elements_next.remove(self)
                # self.delElement(pe)
        return self

    def set_ro(self, new_val):
        '''
        Установка количества выходных патрубков.
        Если в данный момент к элементу подключено больше элементов на выход, чем новое ro, то лишние отбрасываются.
        '''
        self.ro = new_val
        if self.ro<len(self.elements_next):
            num_to_drop=len(self.elements_next)-self.ro
            for _ in range(num_to_drop):
                ne = self.elements_next.pop(0)
                ne.elements_previous.remove(self)
                # self.delElement(ne)

        return self

    def add_links(self, elements_previous, elements_next):
        '''
        Добавляет ссылки на элементы переданных списков.
        Важно! Если у элемента недостаточно слотов для подключения, они не подключаются!
        '''
        for elmnt_next in elements_next:
            self.append(elmnt_next)
        for elmnt_prev in elements_previous:
            elmnt_prev.append(self)
        return self

    # Прямая установка значений
    def get_h(self, d=1):
        '''
        Установка потери напора
            Выход:
                float: текущее значение потери напора для данного элемента. 
                Равно (S*n*q^2) / d
        '''
        self.h = (self.s * self.n * self.q**2) / d
        return self.h

    def get_H_out(self, approved_H=120):
        '''
        Установка напора на выходе из элемента.
            Выход:
                float: текущее значение напора на выходе из элемента.
                Равно H_in + h_add - h - z
        '''
        new_H = self.H_in + self.H_add - self.get_h() - self.z

        try:
            if new_H>approved_H:
                raise ValueError(f"Напор не может быть выше {approved_H}")
        except TypeError as e:
            raise ValueError(f"Напор не может быть выше {approved_H}")

        self.H_out = new_H
        return self.H_out

    def get_q_out(self):
        '''
        Возвращает расход на выходе из элемента
            Выход:
                float - расход на выходе из элемента. Рассчитывается в соответствии с указанной функцией q_out()
        '''
        return self.q_out(self)

    def set_H_add(self, H_add):
        '''
        Устанавливает дополнительный напор для текущего элемента
            Вход:
                H_add=float: дополнительный напор, м
        '''
        self.H_add = H_add      

    def get_L(self):
        '''
        Возвращает длину элемента
        '''
        self.L = self.n * self.l
        return self.L

    # Рекурсивная установка значений
    def set_H_in(self, H_in):
        '''
        Устанавливает напор на входе для текущего элемента, 
        а также далее запускает рекурсивный перерасчет напоров 
        для всех следующих после текущего элементов
            Вход:
                H_in=float: напор на входе в элемент, м
        '''
        self.H_in = H_in
        for elmnt in self.elements_next:
            elmnt.set_H_in(self.get_H_out())

    def set_q_zero(self):
        '''
        Устанавливает нулевой расход для текущего элемента, 
        а также далее запускает рекурсивный перерасчет расходов 
        для всех предыдущих относительно текущего элементов. \n
        Используется для очищения значений расходов при расчете.
        '''
        self.q=0
        for elmnt in self.elements_previous:
            elmnt.set_q_zero()

    def set_q(self, q):
        '''
        Устанавливает расход для текущего элемента,
        а также далее запускает рекурсивный перерасчет расходов 
        для всех предыдущих относительно текущего элементов.\n
        Для каждого элемента происходит суммирование в том случае,
        если элемент является водосборником \n
        Если к элементу подключено несколько других элементов на вход
        Расход к ним разделяется поровну (в данной реализации).
        '''
        self.q+=q
        for elmnt in self.elements_previous:
            elmnt.set_q(q/len(self.elements_previous))
            # Тут нужно разобраться
            # if elmnt.type == 0:
            #     elmnt.H_add = self.h + self.H_in

In [58]:
# Создаем модель НРС:
model = NRS_Model('Модель от одного насоса к одному стволу с новым расчетом потерь напора')

# Создаем элементы модели:
pump   = Element('Н1', EType.PUMP, H_add=40)                                       # Насос
hose   = Element('РРЛ Р1-1', EType.CONNECTOR, s = ss51, n=5)                   # рабочая рукавная линия
nozzle = Element('Ствол 1', EType.NOZZLE, p = p_10, q_out = q_out_nozzle)          # ствол №1


# Соединяем элементы модели вместе в НРС:
# Магистральная часть:
pump.append(hose).append(nozzle)

# Добавляем обозревателей:
NRS_Observer_E(pump, ['q', 'H_add'])             # Для насоса
NRS_Observer_E(hose, ['q', 'h']) 
NRS_Observer_E(nozzle, ['q', 'H_in'])            # Для ствола


# Строим модель:
model = model.build(pump, interpretate=True)

NRS_Revision.print_model_elements(model)

Модель от одного насоса к одному стволу с новым расчетом потерь напора
all:
  Н1
  РРЛ Р1-1
  Ствол 1
in:
  Н1
out:
  Ствол 1


In [59]:
# Активируем обозревателей
model = model.observersInit()
# Шаг моделирования
model.drop_q()
# Напор на насосе
pump.set_H_add(85)

model, r = model.calc(accuracy=0.05)
print('Производительность насоса', pump.q, 'л/с')
print('Потеря напора в рукавной линии', hose.h, 'м')
print('Напор перед стволом', nozzle.H_in, 'м, Производительность ствола', nozzle.q, 'л/с')

ValueError: Напор не может быть выше 120

Тест расчета:

$$
H_н = H_{ств} + h_{р} = H_{ств} + s * n * Q^2 = 40 + 0.09 * 5 * 10^2 = 40 + 44.9 = 85 м
$$

In [64]:
nozzle.history()['q']

[14.57737973711325, (3.1558425668494415e-16+5.153882032022069j)]

In [60]:
hose.history()['h']

[0.0, 95.62499999999997]

In [68]:
p_10 * pow(85, 0.5)

14.57737973711325

Это ничего не дает. Нужно икать другой подход!

Посмотри на ноутбук @model_tessts.ipynb и подумай как можно было бы решить проблему которую я пытаюсь в ней решить. Суть проблемы, что при начальных итерациях значения потерь напор в рукавных линиях могут оказываться чрезмерными, что приводит к невозможности расчета таким способом, хотя расчет исходя из расчета обратным способом (от стволов к насосу) принципиально имеет решение.